# M1 Plan A — yolov11l + SAHI Tile + 100ep (단일학습 0.9 도전)

**전략 (모두 단일학습 차원):**
- 모델: **yolov11l** (47M, M 대비 +3~5 mAP)
- 데이터: **SAHI 타일링 + 원본 val/test** (28K train + 3.5K val)
- imgsz: 640 (타일 사이즈)
- batch: 16 (A100)
- epochs: 100, patience 30
- multi-stage: Stage 1 80ep + Stage 2 fine-tune 20ep

**Drive 준비 (C 계정 = drone_inspect_A 폴더):**
1. `structural_tiled.zip` (2.36GB)

**예상 시간 (A100 batch=16):**
- Stage 1 80ep + Stage 2 20ep = ~10~12시간

**예상 결과: mAP50 0.85~0.93 (0.9 도전)**

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu numpy pillow

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ C 계정: drone_inspect_A (공유받은 폴더 바로가기) ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect_A')
ZIP_NAME = 'structural_tiled.zip'

WORK = Path('/content/m1_plan_a')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
assert zip_path.exists(), f'{zip_path} 없음. drone_inspect_A에 structural_tiled.zip 업로드 필요'
ds_dir = WORK / 'structural_tiled'
if not (ds_dir / 'images' / 'val').exists():
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting structural_tiled.zip ...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')

for s in ['train', 'val', 'test']:
    p = ds_dir / 'images' / s
    if p.exists():
        print(f'  {s}: {len(list(p.glob("*")))} images')

In [ ]:
yaml_text = f'''path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: crack
  1: waterproof_defect
  2: caulking_defect
'''
data_yaml = WORK / 'plan_a.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## Drive 자동 저장 + Resume

In [ ]:
AUTOSAVE = DRIVE_DIR / 'm1_plan_a_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)
PROJECT = Path('/content/runs/m1_plan_a')
PROJECT.mkdir(parents=True, exist_ok=True)

import time, threading
stop_flag = [False]
def autosave():
    while not stop_flag[0]:
        time.sleep(300)
        for s in ['stage1', 'stage2']:
            for src in [PROJECT/s/'weights/last.pt', PROJECT/s/'weights/best.pt']:
                if src.exists():
                    try:
                        shutil.copy2(src, AUTOSAVE / f'{s}_{src.name}')
                    except Exception:
                        pass
        print(f'[autosave] {time.strftime("%H:%M:%S")}')
threading.Thread(target=autosave, daemon=True).start()

## Stage 1: yolov11l + 80ep + lr=1e-3

In [ ]:
from ultralytics import YOLO

print('=== Stage 1 (yolov11l + 640 + 80ep + batch=16) ===')
model_s1 = YOLO('yolo11l.pt')
model_s1.train(
    data=str(data_yaml),
    epochs=80,
    batch=16,
    imgsz=640,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-3,
    lrf=0.01,
    cos_lr=True,
    patience=25,
    warmup_epochs=3,
    close_mosaic=20,
    freeze=0,
    hsv_h=0.015, hsv_s=0.5, hsv_v=0.4,
    degrees=5.0, translate=0.1, scale=0.5,
    shear=2.0, perspective=0.001,
    flipud=0.0, fliplr=0.5,
    mosaic=1.0, mixup=0.15, copy_paste=0.5,
    erasing=0.0,
    multi_scale=0.3,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage1',
    exist_ok=True
)
print('Stage 1 done.')

## Stage 2: Stage1 best → fine-tune (val/test imgsz 1280, lr=1e-5)

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
print(f'=== Stage 2 (lr=1e-5, freeze=10, 20ep) ===')
model_s2 = YOLO(str(stage1_best))
model_s2.train(
    data=str(data_yaml),
    epochs=20,
    batch=16,
    imgsz=640,
    cache='disk',
    workers=4,
    optimizer='AdamW',
    lr0=1e-5,
    lrf=0.01,
    cos_lr=True,
    patience=12,
    warmup_epochs=1,
    close_mosaic=15,
    freeze=10,
    mosaic=0.5, mixup=0.0, copy_paste=0.2,
    save_period=5,
    plots=True,
    project=str(PROJECT),
    name='stage2',
    exist_ok=True
)
stop_flag[0] = True
print('Stage 2 done.')

## ONNX Export + 평가 (val 1280으로 평가, 실사용 시나리오)

In [ ]:
stage1_best = PROJECT / 'stage1' / 'weights' / 'best.pt'
stage2_best = PROJECT / 'stage2' / 'weights' / 'best.pt'
best_path = stage2_best if stage2_best.exists() else stage1_best
print(f'best.pt: {best_path}')
best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')

print('\n=== Val @ imgsz=640 (학습 사이즈) ===')
metrics_640 = best_model.val(data=str(data_yaml), imgsz=640, batch=16)
print(f'  mAP50: {metrics_640.box.map50:.4f}')
print(f'  mAP50-95: {metrics_640.box.map:.4f}')

print('\n=== Val @ imgsz=1280 (실제 운영 사이즈) ===')
metrics_1280 = best_model.val(data=str(data_yaml), imgsz=1280, batch=8)
print(f'  mAP50: {metrics_1280.box.map50:.4f}')
print(f'  mAP50-95: {metrics_1280.box.map:.4f}')
print(f'  precision: {metrics_1280.box.mp:.4f}')
print(f'  recall:    {metrics_1280.box.mr:.4f}')
best_map50 = max(metrics_640.box.map50, metrics_1280.box.map50)
print(f'\n0.9? {"YES" if best_map50 >= 0.9 else "NO"}')

OUT = DRIVE_DIR / 'm1_plan_a_results'
OUT.mkdir(parents=True, exist_ok=True)
shutil.copy2(onnx_path, OUT / 'm1_yolo_structural.onnx')
shutil.copy2(best_path, OUT / 'm1_plan_a_best.pt')
for s in ['stage1', 'stage2']:
    rcsv = PROJECT / s / 'results.csv'
    if rcsv.exists():
        shutil.copy2(rcsv, OUT / f'{s}_results.csv')
print('Saved:', OUT)